In [0]:
# Bucketing

In [0]:
import pyspark.sql.functions as F

In [0]:
# Disable Adaptive Query Execution (AQE)
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "false")
# Disable auto broadcast join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [0]:

df = (
    spark.range(start=1, end=1_00_000+1, numPartitions=5)
    .select(F.col("id"), F.rand(10).alias("value"))
)

df.display()

In [0]:
df.write.format("parquet").mode("overwrite").saveAsTable("non_bucketed_table_1")

In [0]:
df.write.format("parquet").mode("overwrite").saveAsTable("non_bucketed_table_2")

In [0]:
non_bucketed_table_1 = spark.table("non_bucketed_table_1")
non_bucketed_table_2 = spark.table("non_bucketed_table_2")

In [0]:
display(non_bucketed_table_1)

In [0]:
display(non_bucketed_table_2)

In [0]:
non_bucketed_join = (
    non_bucketed_table_1
    .join(
        non_bucketed_table_2,
        on=["id"],
        how="left"
    )
)

non_bucketed_join.display()

In [0]:
df.write.format("parquet").mode("overwrite").bucketBy(4, "id").sortBy("id").saveAsTable("bucketed_table_1")
df.write.format("parquet").mode("overwrite").bucketBy(4, "id").sortBy("id").saveAsTable("bucketed_table_2")

In [0]:
bucketed_table_1 = spark.table("bucketed_table_1")
bucketed_table_2 = spark.table("bucketed_table_2")

In [0]:
bucketed_join = (
    bucketed_table_1
    .join(
        bucketed_table_2,
        on=["id"],
        how="left"
    )
)

non_bucketed_join.display()

In [0]:
dbutils.fs.ls("dbfs:/user/hive/warehouse/bucketed_table_1")

In [0]:
%fs
ls dbfs:/user/hive/warehouse/bucketed_table_1

In [0]:
%fs
ls dbfs:/user/hive/warehouse/non_bucketed_table_1

In [0]:
%sql

select b1.id, b2.value from bucketed_table_1 b1
left join bucketed_table_2 b2
on b1.id = b2.id
